# Séance 8 · Raconter : un tableau de bord et un pitch · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

À la séance 7, tu as mené l'enquête. Aujourd'hui, tu la **racontes** : à quelqu'un qui n'a pas le temps, qui ne lira jamais ton notebook, et qui doit décider. Tu apprends à écrire un titre qui donne la réponse, à faire des graphiques interactifs avec Plotly, à les assembler en **tableau de bord**, à voir comment Streamlit en fait une page web, et à présenter le tout en 2 minutes avec le gabarit **constat → preuve → recommandation**.

Ce notebook tourne dans **Google Colab** (rien à installer). Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter.

**Livrable de la séance** : un mini-dashboard, et le pitch de 2 minutes que tu présentes comme devant un client.

## Préparation

Les deux jeux de données de la séance 7 : **Tips** (244 additions d'un restaurant) et les **Pokémon**. Plotly (`px`) fait les graphiques interactifs : survole, zoome, clique sur la légende.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

URL_TIPS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    tips = pd.read_csv(URL_TIPS)
    pokemon = pd.read_csv(URL_POKEMON)
    print("Tips :", tips.shape, "· Pokémon :", pokemon.shape)
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger les fichiers :", erreur)
tips.head()

In [ ]:
# Vérification des exercices : ✅ si c'est juste, ❌ sinon, sans jamais donner la réponse.
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = booléen, ou fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as erreur:
        print(f"❌ {nom} : erreur pendant la vérification → {erreur}")
        return False
    print(f"✅ {nom}" if ok else f"❌ {nom} : pas encore, réessaie !")
    return ok

## 1. Raconter, c'est répondre

Ton client n'a pas fait l'enquête. Il ne veut pas savoir **ce que tu as fait**, il veut savoir **ce que tu as trouvé** et **ce qu'il doit faire**. Trois règles :

- un graphique = **une** idée ;
- le **titre donne la réponse** : pas « Pourboire selon le jour », mais « Le dimanche, les pourboires sont 9 % plus gros » ;
- on finit par une **recommandation** que le client peut appliquer demain.

Le même graphique, deux titres : lequel aide le patron du restaurant ?

In [ ]:
par_jour = tips.groupby("day")["tip"].mean().round(2).sort_values()
par_jour

À gauche, le titre décrit. À droite, il répond.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, titre in zip(axes, ["Pourboire moyen par jour", "Le dimanche, les pourboires sont les plus gros"]):
    par_jour.plot(kind="bar", ax=ax, color="tab:blue", title=titre)
    ax.set_ylabel("Pourboire moyen ($)"); ax.set_ylim(0, 4)
plt.show()

### À toi : des titres qui répondent

Réécris ces trois titres pour qu'ils donnent la réponse. Les chiffres sont dans la cellule : calcule-les, puis écris un titre par ligne dans la liste `titres` (chacun doit contenir un chiffre).

1. « Pourboire selon le moment » (midi ou soir) ;
2. « Taille des tables » (le nombre de convives le plus fréquent) ;
3. « Fumeurs et non-fumeurs » (le pourboire moyen de chacun).

In [ ]:
# À toi
titres = ["...", "...", "..."]

In [ ]:
verifier("Trois titres avec un chiffre", len(titres) == 3 and all(any(ch.isdigit() for ch in t) for t in titres))

<details><summary>Solution</summary>

```python
print(tips.groupby("time")["tip"].mean().round(2))
print(tips["size"].value_counts().idxmax())
print(tips.groupby("smoker")["tip"].mean().round(2))
titres = ["Le soir, le pourboire moyen dépasse celui du midi : 3,10 $ contre 2,73 $",
          "La plupart des tables sont de 2 personnes",
          "Fumeurs ou non, le pourboire moyen est presque le même : 3,01 $ contre 2,99 $"]
```

</details>

## 2. Des graphiques interactifs avec Plotly

Avec **Plotly**, les graphiques sont interactifs : survole un point, zoome, clique sur la légende pour masquer une catégorie. `plotly.express` (abrégé `px`) fait un graphique en une ligne.

In [ ]:
fig = px.scatter(tips, x="total_bill", y="tip", color="day", size="size", hover_data=["time", "smoker"],
                 title="Pourboire selon l'addition (survole les points !)", labels={"total_bill": "Addition ($)", "tip": "Pourboire ($)"})
fig.show()

## 3. Un sous-graphique par catégorie, et un graphique qui bouge

`facet_col` fait un sous-graphique par catégorie, en une seule ligne.

In [ ]:
# facet_col : un sous-graphique par catégorie, en une seule ligne
fig = px.box(tips, x="day", y="tip", color="smoker", facet_col="time", title="Pourboires par jour, fumeurs ou non, midi et soir")
fig.show()

`animation_frame` fait un graphique qui évolue : ici, génération par génération. Appuie sur ▶.

In [ ]:
# animation_frame : un graphique qui évolue (ici, génération par génération). Appuie sur ▶
fig = px.scatter(pokemon.sort_values("Generation"), x="Attack", y="Defense", color="Type 1", hover_name="Name",
                 animation_frame="Generation", range_x=[0, 200], range_y=[0, 250],
                 title="Attaque vs Défense, génération par génération")
fig.show()

## 4. Un tableau de bord

Un **tableau de bord** (*dashboard*), c'est une page qui rassemble 3 ou 4 graphiques qui répondent aux questions d'un client. On le construit pièce par pièce avec `make_subplots`, et un **menu déroulant** (`updatemenus`) choisit ce qu'on affiche.

D'abord, les outils et les chiffres du premier graphique : une stat moyenne par type.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

moyennes = pokemon.groupby("Type 1")[["HP", "Attack", "Defense", "Speed"]].mean().round(1)
generations = pokemon["Generation"].value_counts().sort_index()

Une grille vide d'une ligne et trois colonnes, avec un titre par case.

In [ ]:
dash = make_subplots(rows=1, cols=3, subplot_titles=("Stat moyenne par type", "Attaque vs Défense", "Nombre par génération"))

Case 1 : une barre par stat, une seule visible à la fois (le menu choisira laquelle).

In [ ]:
for i, stat in enumerate(moyennes.columns):
    dash.add_trace(go.Bar(x=moyennes.index, y=moyennes[stat], name=stat, visible=(i == 0)), row=1, col=1)

Cases 2 et 3 : le nuage attaque / défense, et le nombre de Pokémon par génération.

In [ ]:
dash.add_trace(go.Scatter(x=pokemon["Attack"], y=pokemon["Defense"], mode="markers", text=pokemon["Name"],
                          marker=dict(opacity=0.4), name="Pokémon"), row=1, col=2)
dash.add_trace(go.Bar(x=generations.index, y=generations.values, name="Par génération"), row=1, col=3)

Le menu : un bouton par stat. Chaque bouton dit quelles courbes afficher (les 4 barres de la case 1, puis les 2 autres graphiques, toujours visibles).

In [ ]:
boutons = [dict(label=stat, method="update",
                args=[{"visible": [j == i for j in range(4)] + [True, True]}]) for i, stat in enumerate(moyennes.columns)]
dash.update_layout(updatemenus=[dict(buttons=boutons, x=0.0, y=1.25)], showlegend=False, height=420,
                   title="Mini-dashboard Pokémon : choisis la stat dans le menu")

On affiche le tableau de bord : essaie le menu.

In [ ]:
dash.show()

**Exercice** : ajoute un 4e graphique au dashboard (par exemple un histogramme de `Total`, `go.Histogram`) en passant à `cols=4`. N'oublie pas d'ajouter un `True` dans la liste `visible` de chaque bouton.

In [ ]:
# À toi

In [ ]:
verifier("Quatre graphiques, dont un histogramme", lambda: any(t.type == "histogram" for t in dash.data)
         and len({t.xaxis for t in dash.data}) == 4)

<details><summary>Solution</summary>

```python
dash = make_subplots(rows=1, cols=4, subplot_titles=("Stat moyenne", "Att. vs Déf.", "Par génération", "Total"))
for i, stat in enumerate(moyennes.columns):
    dash.add_trace(go.Bar(x=moyennes.index, y=moyennes[stat], name=stat, visible=(i == 0)), row=1, col=1)
dash.add_trace(go.Scatter(x=pokemon["Attack"], y=pokemon["Defense"], mode="markers", marker=dict(opacity=0.4)), row=1, col=2)
dash.add_trace(go.Bar(x=generations.index, y=generations.values), row=1, col=3)
dash.add_trace(go.Histogram(x=pokemon["Total"]), row=1, col=4)
boutons = [dict(label=s, method="update", args=[{"visible": [j == i for j in range(4)] + [True, True, True]}])
           for i, s in enumerate(moyennes.columns)]
dash.update_layout(updatemenus=[dict(buttons=boutons, x=0, y=1.25)], showlegend=False, height=400)
dash.show()
```
</details>

## 5. Et Streamlit ?

**Streamlit** transforme un script Python en vraie page web avec des boutons et des menus, sans écrire de HTML. Ça ne tourne pas dans Colab directement (il faut un serveur), mais sur ton ordinateur c'est 3 lignes : `pip install streamlit`, un fichier `app.py`, puis `streamlit run app.py`. Voici à quoi ressemblerait notre dashboard Pokémon (à copier dans `app.py`, pas à exécuter ici) :

```python
import streamlit as st
import pandas as pd
import plotly.express as px

st.title("Mon dashboard Pokémon")
pokemon = pd.read_csv("pokemon.csv")
type_choisi = st.selectbox("Choisis un type", sorted(pokemon["Type 1"].unique()))   # un menu déroulant
stat = st.radio("Stat à afficher", ["HP", "Attack", "Defense", "Speed"])            # des boutons

sous_ensemble = pokemon[pokemon["Type 1"] == type_choisi]
st.metric("Nombre de Pokémon", len(sous_ensemble))                                    # un gros chiffre
st.plotly_chart(px.histogram(sous_ensemble, x=stat, title=f"{stat} des Pokémon {type_choisi}"))
st.dataframe(sous_ensemble.sort_values(stat, ascending=False).head(10))               # un tableau
```

À chaque clic dans le menu, Streamlit relance le script et met la page à jour. C'est l'outil des data scientists pour montrer un résultat à un client sans lui envoyer un notebook.

## 6. Projet : ton dashboard et ton pitch

Reprends ta fiche d'enquête de la séance 7 (ou, si tu n'as pas pu la finir, choisis le patron du restaurant et trois questions sur Tips). Ton client n'a pas le temps : il veut 3 graphiques et 2 minutes d'explication.

Consignes :
1. Une question par graphique Plotly (`px.scatter`, `px.bar`, `px.box`, `px.histogram`…), avec un titre qui donne la réponse.
2. Assemble-les en un dashboard (`make_subplots`, ou 3 `px` avec `facet_col` / `animation_frame`).
3. Vérifie tes pièges : axe qui part de zéro ? corrélation présentée comme une cause ? échantillon représentatif ?
4. Prépare ton pitch de 2 minutes avec le gabarit **constat → preuve → recommandation**.

In [ ]:
# Question 1 :

In [ ]:
# Question 2 :

In [ ]:
# Question 3 :

In [ ]:
# Mon dashboard (3 graphiques assemblés)

### Le gabarit de présentation (2 minutes, comme devant un client)

Remplis les trois cases ci-dessous. Une seule idée par graphique, des chiffres concrets, et une recommandation que le client peut appliquer demain.

| Étape | Ce que tu dis | Exemple |
|---|---|---|
| **Constat** (20 s) | Ce que tu as trouvé, en une phrase | « Le dimanche soir, le pourboire moyen atteint 3,26 $ contre 3,00 $ en moyenne, environ 9 % de plus » |
| **Preuve** (60 s) | Le graphique qui le montre, et comment le lire | « Sur ce graphique, chaque point est une addition... » |
| **Recommandation** (40 s) | Ce que le client devrait faire | « Mettre les meilleurs serveurs le dimanche soir » |

In [ ]:
pitch = {
    "client": "...",
    "constat": "...",
    "preuve": "... (quel graphique, quel chiffre)",
    "recommandation": "...",
    "piege_verifie": "... (axe à zéro ? causalité ? biais ?)",
}

Le pitch, tel que tu vas le dire.

In [ ]:
for etape, texte in pitch.items():
    print(f"{etape.upper():16s} {texte}")

## À retenir

- Raconter, c'est **répondre** : un graphique = une idée, et le **titre donne la réponse**
- **Plotly** rend les graphiques interactifs ; `facet_col` = un sous-graphique par catégorie, `animation_frame` = un graphique qui bouge
- `make_subplots` + `updatemenus` = un tableau de bord avec un menu ; **Streamlit** en fait une page web
- Une présentation client tient en trois temps : **constat, preuve, recommandation**
- Avant de présenter, on revérifie les pièges : axe à zéro, causalité, biais

## Pour montrer aux autres

Pendant les 15 dernières minutes, chacun présente son dashboard en 2 minutes chrono, les autres jouent le client. Trois questions guides :

1. Quel est ton constat principal, en une phrase avec un chiffre ?
2. Quel graphique le prouve, et pourquoi as-tu choisi celui-là ?
3. Quel piège as-tu vérifié avant de conclure (causalité, axe, biais) ?

## Liens gratuits

- Galerie Plotly Express : https://plotly.com/python/plotly-express/
- Streamlit, démarrer en 5 minutes : https://docs.streamlit.io/get-started